In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

df = pd.read_excel("../data/processed/BDD_CAPM_nettoyee.xlsx")

print("Dimensions :", df.shape)
df.head()

Dimensions : (22188, 12)


,Age Année,Sexe,Milieu,Région,Circonstance,Sous circonstance,type de produit1,Produit 1,Voie 1,Symptomatologie,Gradation finale,Tranche_age
0,25.0,Féminin,Urbain,Rabat-Salé-Kénitra,Inconnu,Inconnu,Médicament,SEROQUEL,Orale,Oui,Grade 2,16-30 ans
1,20.0,Féminin,Urbain,Casablanca-Settat,Inconnu,Inconnu,Médicament,LAROXYL,Orale,Oui,Grade 2,16-30 ans
2,16.0,Féminin,Urbain,Fès-Meknès,Accidentelle,Accident classique,Médicament,FLUOXETINE,Orale,Oui,Grade 2,16-30 ans
3,2.0,Masculin,Urbain,Fès-Meknès,Accidentelle,Erreur thérapeutique,Médicament,CODETUX,Orale,Oui,Grade 0,0-5 ans
4,32.0,Masculin,Inconnu,Casablanca-Settat,Accidentelle,Effet indésirable,Médicament,MEDICAMENT INCONNU,Inconnu,Oui,Grade 2,31-60 ans


In [2]:
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

variables_a_tester = ['Sexe', 'Milieu', 'Région', 'Circonstance', 'Sous circonstance',
                       'type de produit1', ' Voie 1', 'Symptomatologie ', 'Tranche_age']

resultats = []
for var in variables_a_tester:
    table = pd.crosstab(df[var], df['Gradation finale'])
    chi2, p_value, dof, expected = chi2_contingency(table)
    v = cramers_v(table)
    resultats.append({'Variable': var, 'Chi2': round(chi2, 1), 'p_value': p_value, "V de Cramer": round(v, 3)})

df_resultats = pd.DataFrame(resultats).sort_values("V de Cramer", ascending=False)
print(df_resultats)

            Variable     Chi2       p_value  V de Cramer
7   Symptomatologie   11697.7  0.000000e+00        0.513
5   type de produit1   4540.5  0.000000e+00        0.226
8        Tranche_age   2978.5  0.000000e+00        0.183
4  Sous circonstance   1990.9  0.000000e+00        0.150
2             Région   1873.4  0.000000e+00        0.145
6             Voie 1   1554.1  0.000000e+00        0.132
3       Circonstance    468.4  1.145287e-92        0.084
1             Milieu    293.7  8.880617e-59        0.081
0               Sexe    197.9  1.757836e-38        0.067


In [3]:
from scipy.stats import chi2_contingency, chi2
import numpy as np

def cramers_v(confusion_matrix):
    chi2_stat = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2_stat / (n * (min(r, k) - 1)))

variables_a_tester = ['Sexe', 'Milieu', 'Région', 'Circonstance', 'Sous circonstance',
                       'type de produit1', ' Voie 1', 'Symptomatologie ', 'Tranche_age']

resultats = []
for var in variables_a_tester:
    table = pd.crosstab(df[var], df['Gradation finale'])
    chi2_stat, p_value, dof, expected = chi2_contingency(table)
    v = cramers_v(table)

    # Seuil critique du Chi2 pour p=0.05, à ce degré de liberté (dof)
    seuil_critique = chi2.ppf(0.95, dof)

    resultats.append({
        'Variable': var,
        'Chi2': round(chi2_stat, 1),
        'ddl (degrés de liberté)': dof,
        'Seuil critique (p=0.05)': round(seuil_critique, 1),
        'p_value (notation scientifique)': f"{p_value:.3e}",
        'Significatif ?': "Oui" if chi2_stat > seuil_critique else "Non",
        "V de Cramer": round(v, 3)
    })

df_resultats = pd.DataFrame(resultats).sort_values("V de Cramer", ascending=False)
print(df_resultats.to_string(index=False))

         Variable    Chi2  ddl (degrés de liberté)  Seuil critique (p=0.05) p_value (notation scientifique) Significatif ?  V de Cramer
 Symptomatologie  11697.7                        8                     15.5                       0.000e+00            Oui        0.513
 type de produit1  4540.5                       52                     69.8                       0.000e+00            Oui        0.226
      Tranche_age  2978.5                       16                     26.3                       0.000e+00            Oui        0.183
Sous circonstance  1990.9                       48                     65.2                       0.000e+00            Oui        0.150
           Région  1873.4                       48                     65.2                       0.000e+00            Oui        0.145
           Voie 1  1554.1                       24                     36.4                       0.000e+00            Oui        0.132
     Circonstance   468.4                       